In [2]:
# Part 1: Import & Directory Setup

import os
import re
import time
import urllib.request
from datetime import date
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, wait
from pdfminer.high_level import extract_text
from io import BytesIO
import logging
import shutil
import threading
import csv

# Configuration
BASE_DIR = os.getcwd()
PATH_PDF = os.path.join(BASE_DIR, 'PDF', 'Narkotika')
PATH_OUTPUT = os.path.join(BASE_DIR, 'data', 'raw')
LOG_DIR = os.path.join(BASE_DIR, 'logs')
LOG_PATH = os.path.join(LOG_DIR, 'scraper_narkotika.log')
CSV_PATH = os.path.join(BASE_DIR, "data", "putusan.csv")

processed_files = 0
file_lock = threading.Lock()
MAX_PATH_LENGTH = 260

def validate_path(path):
    if len(path) > MAX_PATH_LENGTH:
        raise ValueError(f"Path {path} exceeds Windows maximum length of {MAX_PATH_LENGTH} characters")
    return path

# Ensure dirs
for path in [PATH_PDF, PATH_OUTPUT, LOG_DIR, os.path.dirname(CSV_PATH)]:
    validate_path(path)
    os.makedirs(path, exist_ok=True)

# Init logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, mode="a"), logging.StreamHandler()]
)
logging.info("Logging initialized.")

# Part 2: Utility Functions (Open Page, PDF URL, Download)

def open_page(link):
    for i in range(3):
        try:
            r = requests.get(link)
            r.raise_for_status()
            return BeautifulSoup(r.text, "lxml")
        except Exception as e:
            logging.warning(f"Attempt {i+1} failed for {link}: {e}")
            time.sleep(5)
    logging.error(f"Could not open {link}")
    return None

def get_pdf_url(soup):
    try:
        pdf_link = soup.find("a", href=re.compile(r"/pdf/"))["href"]
        if not pdf_link.startswith("http"):
            pdf_link = "https://putusan3.mahkamahagung.go.id" + pdf_link
        return pdf_link
    except:
        return None

def is_url_already_processed(url, path_pdf):
    processed = [f for f in os.listdir(path_pdf) if f.endswith(".pdf")]
    return any(url.split("/")[-1] in f for f in processed)

def download_pdf(url, path_pdf, keyword_url):
    try:
        file = urllib.request.urlopen(url)
        file_name = file.info().get_filename() or url.split("/")[-1]
        keyword = "NarkotikaPsikotropika" if "narkotika" in keyword_url.lower() else "case"
        file_name = re.sub(r"[^\w\-\.]", "", file_name.replace(".pdf", ""))
        file_name = f"{file_name}{keyword}{date.today().strftime('%Y-%m-%d')}.pdf"
        save_path = os.path.join(path_pdf, file_name)
        with open(save_path, "wb") as out_file:
            out_file.write(file.read())
        logging.info(f"Downloaded: {file_name}")
        return file_name
    except Exception as e:
        logging.error(f"Failed download {url}: {e}")
        return None

# Part 3: Metadata & Catatan Amar Extractor

def parse_catatan_amar(soup):
    """Ambil Amar Putusan + Barang Bukti dari Catatan Amar"""
    try:
        amar_section = soup.find("td", string=re.compile("Catatan Amar", re.I)).find_next("td")
        full_amar = amar_section.get_text(separator=" ", strip=True)

        # Cari trigger phrase umum
        bb_match = re.search(
            r"(Memerintahkan|Menetapkan).?barang bukti berupa:(.?)(?:Membebankan|Tanggal|$)",
            full_amar, re.S | re.I
        )
        if bb_match:
            barang_bukti = bb_match.group(2).strip()
        else:
            # fallback: cari kalimat yang ada kata "barang bukti"
            bb_fallback = re.search(r"([^.]barang bukti[^.])", full_amar, re.I)
            barang_bukti = bb_fallback.group(1).strip() if bb_fallback else ""

        return full_amar, barang_bukti
    except:
        return "", ""

def extract_metadata(case_url):
    soup = open_page(case_url)
    if not soup:
        return None

    def safe_get(label):
        el = soup.find("td", string=re.compile(label, re.I))
        if el and el.find_next("td"):
            return el.find_next("td").get_text(strip=True)
        return ""

    amar_putusan, barang_bukti = parse_catatan_amar(soup)

    metadata = {
        "No": "",  # auto increment nanti
        "No Putusan": safe_get("Nomor"),
        "Lembaga Peradilan": safe_get("Lembaga Peradilan"),
        "Barang Bukti": barang_bukti,
        "Amar Putusan": amar_putusan,
        "Link": case_url
    }
    return metadata

# Part 4: Data Extraction (Combine PDF + Metadata)

def extract_data(link, keyword_url, max_files, csv_writer, counter):
    global processed_files
    with file_lock:
        if processed_files >= max_files:
            return False

    if is_url_already_processed(link, PATH_PDF):
        logging.info(f"Skipping duplicate {link}")
        return True

    soup = open_page(link)
    if not soup:
        return True

    # Get PDF
    link_pdf = get_pdf_url(soup)
    if not link_pdf:
        logging.info(f"No PDF found {link}")
        return True

    # Download
    file_name = download_pdf(link_pdf, PATH_PDF, keyword_url)

    # Metadata
    metadata = extract_metadata(link)
    if metadata and file_name:
        with file_lock:
            processed_files += 1
            metadata["No"] = processed_files
            csv_writer.writerow(metadata)
            logging.info(f"Saved metadata + PDF {processed_files}/{max_files}")
            if processed_files >= max_files:
                return False
    return True

# Part 5: Page Processing

def run_process(keyword_url, page, sort_page, max_files, csv_writer, counter):
    global processed_files
    if processed_files >= max_files:
        return False

    link = f"{keyword_url}&page={page}" if keyword_url.startswith("https") else f"https://putusan3.mahkamahagung.go.id/search.html?q={keyword_url}&page={page}"
    if sort_page:
        link += "&obf=TANGGAL_PUTUS&obm=desc"

    soup = open_page(link)
    if not soup:
        return False

    links = soup.find_all("a", {"href": re.compile("/direktori/putusan")})
    for l in links:
        full_link = l["href"]
        if not full_link.startswith("http"):
            full_link = "https://putusan3.mahkamahagung.go.id" + full_link
        cont = extract_data(full_link, keyword_url, max_files, csv_writer, counter)
        if not cont:
            return False
    return True

# Part 6: Main Scraper

def run_scraper(url=None, max_files=3):
    global processed_files
    if not url or not url.startswith("https://"):
        logging.error("Invalid URL")
        return

    soup = open_page(url)
    if not soup:
        return

    try:
        last_page = int(soup.find_all("a", {"class": "page-link"})[-1].get("data-ci-pagination-page"))
    except:
        last_page = 1

    logging.info(f"Scraping {last_page} pages")

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["No", "No Putusan", "Lembaga Peradilan", "Barang Bukti", "Amar Putusan", "Link"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        with ThreadPoolExecutor(max_workers=1) as executor:
            futures = []
            for page in range(1, last_page + 1):
                if processed_files >= max_files:
                    break
                futures.append(executor.submit(run_process, url, page, True, max_files, writer, processed_files))
            wait(futures)

    logging.info(f"Scraper done. Total {processed_files} cases saved to CSV")

# Part 7: PDF Processing

def extract_pdf_text(pdf_path):
    try:
        with open(pdf_path, "rb") as f:
            return extract_text(BytesIO(f.read()))
    except:
        return ""

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def save_text_file(text, index, path):
    if not text:
        return None
    file_name = f"case_{index:03d}.txt"
    with open(os.path.join(path, file_name), "w", encoding="utf-8") as f:
        f.write(text)
    return file_name

def process_pdfs(max_files=3):
    pdf_files = [f for f in os.listdir(PATH_PDF) if f.endswith(".pdf")]
    for idx, pdf in enumerate(pdf_files, 1):
        if idx > max_files:
            break
        text = clean_text(extract_pdf_text(os.path.join(PATH_PDF, pdf)))
        if text:
            save_text_file(text, idx, PATH_OUTPUT)

# Part 8: Main Runner

def main():
    url = "https://putusan3.mahkamahagung.go.id/search.html?q=Pidana%20Khusus&jenis_doc=&cat=3c40e48bbab311301a21c445b3c7fe57&jd=&tp=&court=098167PN337&t_put=&t_reg=&t_upl=&t_pr="
    run_scraper(url=url, max_files=3)
    process_pdfs(max_files=3)

if __name__ == "_main_":
    main()

2025-09-22 23:26:36,702 - INFO - Logging initialized.
